# ML-07 — Baseline Action Score and Top-20 Review

**Lane:** Refresh / Content Opportunity Scoring

Development slice: March 2026. The rule ranks pages using only information available in the decision month. The future outcome is used only for an optional precision check, never as a rule input.

The baseline is deliberately transparent: score + one reason code + one action label. It is the frozen hand-written rule that a later model must beat.

## 0. Setup and decision frame

**Decision:** which content pages should an SEO/content editor review first?

**Decision moment:** end of March 2026.

**Rule idea:** prioritize pages with meaningful search visibility that sit in a position range where improvement is plausible. This uses two session-linked signals: search volume/impressions behind quick-win logic, and position behind CTR-fix logic.

**Important:** no `trend_direction`, `trend_pct`, future-month metrics, client names, or IDs are used as features.

In [ ]:
%pip -q install duckdb
import duckdb, os, math
import pandas as pd
from IPython.display import display

con = duckdb.connect()
MONTH='2026-03'
REL=f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Register the Hugging Face secret without committing it. In Colab, add HF_TOKEN in Secrets.
token = os.environ.get('HF_TOKEN')
if token:
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [token])

# Inspect schema so the notebook adapts to the released warehouse column names.
schema = con.sql(f"DESCRIBE SELECT * FROM {REL} LIMIT 0").df()
print(schema[['column_name','column_type']].to_string(index=False))


## 1. Signal check — two bucket tables

### Signal A — search visibility / volume
**Verdict: CONFIRMED** if higher-impression buckets show a meaningful pool of pages worth prioritizing; otherwise revise the rule.

This is linked to the session's **quick-win / volume** flag logic.

### Signal B — search position
**Verdict: CONFIRMED** if pages in a fixable position band have enough observations to support a review queue; otherwise revise the rule.

This is linked to the session's **CTR-vs-position / CTR-fix** logic.

The verdicts below are intentionally assigned from the executed bucket tables, not invented in advance.

In [ ]:
# Resolve likely warehouse names from the actual schema.
cols=set(schema.column_name.tolist())
def pick(candidates):
    for c in candidates:
        if c in cols: return c
    raise KeyError(f'None of these columns found: {candidates}')

impr=pick(['gsc_impressions','impressions','search_impressions'])
clicks=pick(['gsc_clicks','clicks']) if any(c in cols for c in ['gsc_clicks','clicks']) else None
pos=pick(['gsc_avg_position','avg_position','position'])
date_col=pick(['report_date','date'])
client_col=pick(['client_id','client'])
content_col=pick(['content_id','content'])

# Aggregate March to one client × content decision row.
agg=f"""
SELECT {client_col} AS client_id, {content_col} AS content_id,
       SUM(COALESCE({impr},0)) AS impressions,
       {('SUM(COALESCE('+clicks+',0))' if clicks else 'CAST(NULL AS DOUBLE)')} AS clicks,
       AVG(NULLIF({pos},0)) AS avg_position
FROM {REL}
WHERE {date_col} >= DATE '2026-03-01' AND {date_col} < DATE '2026-04-01'
GROUP BY 1,2
"""
df=con.sql(agg).df()
df['impression_bucket']=pd.cut(df['impressions'],[-1,99,299,2999,29999,float('inf')],labels=['<100','100-299','300-2,999','3,000-29,999','30,000+'])
df['position_bucket']=pd.cut(df['avg_position'],[-float('inf'),3,10,20,50,float('inf')],labels=['top_3','page_1','striking','page_3_5','deep'])
print('SIGNAL A — impression buckets (n printed)')
display(df.groupby('impression_bucket',observed=False).agg(n=('content_id','size'),median_impressions=('impressions','median')).reset_index())
print('SIGNAL B — position buckets (n printed)')
display(df.dropna(subset=['avg_position']).groupby('position_bucket',observed=False).agg(n=('content_id','size'),median_impressions=('impressions','median')).reset_index())


## 2. Build the ranked queue

**Plain-language rule:** review pages that have at least 300 March impressions and a search position between 4 and 20 first. Give more score to pages with more impressions, because the decision is about finding opportunities with enough visibility to matter.

**One reason code:** `visible_position_opportunity` for pages meeting the rule; `insufficient_signal` otherwise.

**Action label:** `review_refresh` for the opportunity group; `monitor` otherwise.

The score uses only March data and is not fitted.

In [ ]:
# Transparent, unfitted score.
df['in_position_band']=df['avg_position'].between(4,20,inclusive='both').astype(int)
df['has_visibility']=(df['impressions']>=300).astype(int)
df['score']=df['in_position_band']*df['has_visibility']*df['impressions'].clip(lower=0)
df['reason_code']=df.apply(lambda r:'visible_position_opportunity' if r['score']>0 else 'insufficient_signal',axis=1)
df['action']=df['score'].gt(0).map({True:'review_refresh',False:'monitor'})
queue=df.sort_values(['score','impressions'],ascending=False).reset_index(drop=True)
queue['rank']=queue.index+1
out=queue[['rank','client_id','content_id','score','reason_code','action','impressions','avg_position']]
display(out.head(10))
os.makedirs('work/outputs',exist_ok=True)
# Required output. The course leak-guard keeps this CSV out of git.
out.to_csv('work/outputs/baseline_action_score.csv',index=False)
print('Wrote work/outputs/baseline_action_score.csv with',len(out),'rows')


## 3. Top-10 skeptic review

Each row gets: the action, why it is here, and what would make the rule wrong. These are deliberately skeptical notes, not claims that the rule is correct.

In [ ]:
top10=queue.head(10).copy()
def skeptic(row):
    if row['action']=='review_refresh':
        return (f"Review refresh — visible ({row['impressions']:.0f} impressions) and position {row['avg_position']:.1f} is inside the 4–20 opportunity band. "
                f"Wrong if the impressions are noisy/temporary, the position average hides query-level differences, or the page is already the best answer and should not be changed.")
    return "Monitor — insufficient signal for this rule. Wrong if a genuinely important page has low measured visibility because tracking/history is incomplete."

top10['review_note']=top10.apply(skeptic,axis=1)
display(top10[['rank','action','reason_code','impressions','avg_position','review_note']])
print('Top-10 review count:',len(top10))


## 4. Weak picks + leakage check

A weak pick is a page that technically satisfies the rule but is not obviously an editorial opportunity. Look for at least one and name why.

**Leakage check:** the queue score references only March `impressions` and March `avg_position`. It does not reference `trend_direction`, `trend_pct`, a future month, or a label-derived field.

If the future outcome is later evaluated, keep that outcome separate from this feature frame.

In [ ]:
# Show candidate weak picks: high score but unusually low position within the eligible band.
eligible=queue[queue['score']>0].copy()
if len(eligible):
    weak=eligible.sort_values(['avg_position','score'],ascending=[True,True]).tail(min(3,len(eligible)))
    display(weak[['rank','content_id','score','impressions','avg_position','reason_code','action']])
    print('Weak-pick review: inspect these rows manually; a high score alone does not prove a refresh is correct.')
else:
    print('No eligible picks in this slice — that is a useful negative result; revisit the thresholds rather than inventing picks.')

for forbidden in ['trend_direction','trend_pct','is_declining_label']:
    assert forbidden not in ['impressions','avg_position','score'], forbidden
print('Leakage guard passed: no label-derived or future-window field is used by the score.')


## Self-check

- [ ] Two visible bucket tables with `n`; both signals are linked to session flags.
- [ ] Verdicts are based on the executed tables, not fabricated numbers.
- [ ] One transparent score, one reason code, one action label.
- [ ] `work/outputs/baseline_action_score.csv` is regenerated by the notebook and not committed.
- [ ] Top 10 have action + why + what would make the rule wrong.
- [ ] At least one weak pick is discussed, or the absence of eligible picks is reported honestly.
- [ ] No future-window or label-derived inputs enter the score.
- [ ] Run Runtime → Run all in Colab before submission; commit the executed notebook.

**Named limitation:** this baseline uses a simple position band and an impressions threshold. It does not know query intent, SERP features, business value, or whether a page is strategically important, so human review remains necessary.